In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image
import cv2

# Load the YOLO model
yolo_model = YOLO("yolo11x.pt")

# Choose which patch to apply
patch_type = "sigmoid"  # or "sigmoid"

# Apply the selected patch
if patch_type == "sigmoid":
    import yolo_patch_sigmoid
elif patch_type == "softmax":
    import yolo_patch_softmax

# Image path
image_path = "image.jpg"


In [ ]:
def parse_detection_results(results):
    """
    Extract detection outputs from YOLO-like model results.

    Args:
        results: List of detection results (e.g., from Ultralytics YOLOv8).

    Returns:
        tuple: (xyxy, conf, cls, prob_vectors, names)
            - xyxy: ndarray of shape (N, 4), bounding boxes in [x1, y1, x2, y2] format
            - conf: ndarray of shape (N,), confidence scores
            - cls: ndarray of shape (N,), class IDs
            - prob_vectors: ndarray of shape (N, num_classes), per-class probabilities
            - names: list or dict of class names
            - num_detections: int, number of detections
    """
    result = results[0]  # One image, one result

    xyxy = result.boxes.xyxy.cpu().numpy()         # (N, 4)
    conf = result.boxes.conf.cpu().numpy()         # (N,)
    cls = result.boxes.cls.cpu().numpy()           # (N,)
    prob_vectors = result.boxes.data[:, 6:].cpu().numpy()  # (N, num_classes)
    names = result.names                           # class names
    num_detections = len(xyxy)                    # number of detections

    return xyxy, conf, cls, prob_vectors, names, num_detections


In [ ]:
results = yolo_model.predict(source=image_path, device='cuda:0', conf=0.30, iou=0.40, verbose=False, max_det=10)
xyxy, conf, cls, prob_vectors, names, num_detections = parse_detection_results(results)

In [ ]:
# Load the image
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Plot the image and overlay bounding boxes
plt.figure(figsize=(10, 10))
plt.imshow(image)

for box, class_id, confidence in zip(xyxy, cls, conf.flatten()):
	x1, y1, x2, y2 = box
	label = f"{names[class_id]}: {confidence:.2f}"
	plt.gca().add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, edgecolor='red', linewidth=2, fill=False))
	plt.text(x1, y1 - 10, label, color='red', fontsize=12, backgroundcolor='white')

plt.axis('off')
plt.title("Detection Results")
plt.show()

In [ ]:
# Print the results
print(f"Number of detections: {num_detections}")
print("Bounding boxes (xyxy):", xyxy)
print("Confidence scores:", conf)
print("Class IDs:", cls)


In [ ]:
for prob_vector in prob_vectors:
    print("Sum of probabilities: {:.3f}".format(prob_vector.sum()), end=' ')
    print("\tMax probability: {:.3f}".format(prob_vector.max()), end=' ')
    print("\tClass with max probability:", prob_vector.argmax(), "->", names[prob_vector.argmax()])

### Softmax results 
Sum of probabilities: 1.000 	Max probability: 1.000 	Class with max probability: 57 -> couch
Sum of probabilities: 1.000 	Max probability: 1.000 	Class with max probability: 56 -> chair
Sum of probabilities: 1.000 	Max probability: 1.000 	Class with max probability: 58 -> potted plant
Sum of probabilities: 1.000 	Max probability: 1.000 	Class with max probability: 56 -> chair
Sum of probabilities: 1.000 	Max probability: 0.998 	Class with max probability: 75 -> vase
Sum of probabilities: 1.000 	Max probability: 0.995 	Class with max probability: 56 -> chair
Sum of probabilities: 1.000 	Max probability: 0.975 	Class with max probability: 75 -> vase
Sum of probabilities: 1.000 	Max probability: 0.971 	Class with max probability: 57 -> couch
Sum of probabilities: 1.000 	Max probability: 0.868 	Class with max probability: 56 -> chair

### Sigmoid results
Sum of probabilities: 1.000 	Max probability: 1.000 	Class with max probability: 57 -> couch
Sum of probabilities: 1.000 	Max probability: 1.000 	Class with max probability: 58 -> potted plant
Sum of probabilities: 1.000 	Max probability: 0.999 	Class with max probability: 56 -> chair
Sum of probabilities: 1.000 	Max probability: 0.999 	Class with max probability: 56 -> chair
Sum of probabilities: 1.000 	Max probability: 0.998 	Class with max probability: 75 -> vase
Sum of probabilities: 1.000 	Max probability: 0.991 	Class with max probability: 56 -> chair
Sum of probabilities: 1.000 	Max probability: 0.962 	Class with max probability: 75 -> vase
Sum of probabilities: 1.000 	Max probability: 0.914 	Class with max probability: 57 -> couch
Sum of probabilities: 1.000 	Max probability: 0.783 	Class with max probability: 56 -> chair